# Cod adapdat dupa: [Transformers Fine-Tuning](https://huggingface.co/docs/transformers/training)

In [ ]:
!pip install transformers torch pandas scikit-learn


In [ ]:
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import torch

In [ ]:
# Tokenization
tokenizer = BertTokenizer.from_pretrained('dumitrescustefan/bert-base-romanian-cased-v1')
def tokenize_data(data):
    return tokenizer(data['sentence'].tolist(), padding=True, truncation=True, return_tensors="pt")

# Create DataLoader
def create_dataloader(encodings, labels, batch_size=16):
    dataset = TensorDataset(encodings['input_ids'], encodings['attention_mask'], labels)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/397k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [ ]:
# Load and clean data
horoscop_train = pd.read_csv('/content/drive/MyDrive/licenta/clasificare-bert/clasificare_horoscop_train.tsv', sep='\t')
horoscop_train['class'] = horoscop_train['class'].replace('positiv', 'pozitiv')
horoscop_train.dropna(subset=['sentence', 'class'], inplace=True)

# Split data
train_data, val_data = train_test_split(horoscop_train, test_size=0.1, random_state=42, stratify=horoscop_train['class'])

train_encodings = tokenize_data(train_data)
val_encodings = tokenize_data(val_data)

# Convert labels to tensor
label_map = {'pozitiv': 0, 'negativ': 1, 'neutru': 2}
train_labels = torch.tensor([label_map[label] for label in train_data['class'].tolist()])
val_labels = torch.tensor([label_map[label] for label in val_data['class'].tolist()])

In [ ]:


train_dataloader = create_dataloader(train_encodings, train_labels)
val_dataloader = create_dataloader(val_encodings, val_labels)

# Model and optimizer
model = BertForSequenceClassification.from_pretrained('dumitrescustefan/bert-base-romanian-cased-v1', num_labels=3)
optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 3
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dumitrescustefan/bert-base-romanian-cased-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(50000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:

# Training function
def train(model, train_dataloader, val_dataloader):
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        total_train_accuracy = 0

        for batch in train_dataloader:
            b_input_ids, b_input_mask, b_labels = batch
            b_input_ids, b_input_mask, b_labels = b_input_ids.to(device), b_input_mask.to(device), b_labels.to(device)

            model.zero_grad()
            outputs = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)
            loss = outputs.loss
            logits = outputs.logits
            total_train_loss += loss.item()

            preds = torch.argmax(logits, dim=1).flatten()
            total_train_accuracy += accuracy_score(b_labels.cpu().numpy(), preds.cpu().numpy())

            loss.backward()
            optimizer.step()
            scheduler.step()

        avg_train_loss = total_train_loss / len(train_dataloader)
        avg_train_accuracy = total_train_accuracy / len(train_dataloader)

        model.eval()
        total_val_loss = 0
        total_val_accuracy = 0

        for batch in val_dataloader:
            b_input_ids, b_input_mask, b_labels = batch
            b_input_ids, b_input_mask, b_labels = b_input_ids.to(device), b_input_mask.to(device), b_labels.to(device)

            with torch.no_grad():
                outputs = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)
                loss = outputs.loss
                logits = outputs.logits
                total_val_loss += loss.item()

                preds = torch.argmax(logits, dim=1).flatten()
                total_val_accuracy += accuracy_score(b_labels.cpu().numpy(), preds.cpu().numpy())

        avg_val_loss = total_val_loss / len(val_dataloader)
        avg_val_accuracy = total_val_accuracy / len(val_dataloader)

        print(f"Epoca {epoch + 1} - Pierdere antrenare: {avg_train_loss}, Acuratețe antrenare: {avg_train_accuracy}")
        print(f"Epoca {epoch + 1} - Pierdere validare: {avg_val_loss}, Acuratețe validare: {avg_val_accuracy}")




In [ ]:
# Train model for horoscope
train(model, train_dataloader, val_dataloader)

# Save model
model.save_pretrained("bert_model_horoscop")

# Upload to Hugging Face Hub
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig

model.push_to_hub("iulik-pisik/bert_model_horoscop")
tokenizer.push_to_hub("iulik-pisik/bert_model_horoscop")

Epoca 1 - Pierdere antrenare: 0.46711713516263553, Acuratețe antrenare: 0.8413978494623656
Epoca 1 - Pierdere validare: 0.3959041434255513, Acuratețe validare: 0.8636363636363636
Epoca 2 - Pierdere antrenare: 0.24443070474331097, Acuratețe antrenare: 0.9173387096774194
Epoca 2 - Pierdere validare: 0.25802233984524553, Acuratețe validare: 0.9204545454545454
Epoca 3 - Pierdere antrenare: 0.13798969517391857, Acuratețe antrenare: 0.9623655913978495
Epoca 3 - Pierdere validare: 0.28711428798057814, Acuratețe validare: 0.9090909090909091


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/iulik-pisik/bert_model_horoscop/commit/666d5ace156e68d71bc0fe4a01dad35873a2664d', commit_message='Upload tokenizer', commit_description='', oid='666d5ace156e68d71bc0fe4a01dad35873a2664d', pr_url=None, pr_revision=None, pr_num=None)

In [ ]:
# Load and clean data
vreme_train = pd.read_csv('/content/drive/MyDrive/licenta/clasificare-bert/clasificare_vreme_train.tsv', sep='\t')
vreme_train.dropna(subset=['sentence', 'class'], inplace=True)

# Split data
train_data, val_data = train_test_split(vreme_train, test_size=0.1, random_state=42, stratify=vreme_train['class'])

# Tokenization
train_encodings = tokenize_data(train_data)
val_encodings = tokenize_data(val_data)

# Convert labels to tensor
label_map = {'pozitiv': 0, 'negativ': 1, 'neutru': 2}
train_labels = torch.tensor([label_map[label] for label in train_data['class'].tolist()])
val_labels = torch.tensor([label_map[label] for label in val_data['class'].tolist()])

# Create DataLoader
train_dataloader = create_dataloader(train_encodings, train_labels)
val_dataloader = create_dataloader(val_encodings, val_labels)


# Model and optimizer
model = BertForSequenceClassification.from_pretrained('dumitrescustefan/bert-base-romanian-cased-v1', num_labels=3)
optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 3
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Train model for weather
train(model, train_dataloader, val_dataloader)

# Save model
model.save_pretrained("bert_model_vreme")

# Upload to Hugging Face Hub
model.push_to_hub("iulik-pisik/bert_model_vreme")
tokenizer.push_to_hub("iulik-pisik/bert_model_vreme")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dumitrescustefan/bert-base-romanian-cased-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoca 1 - Pierdere antrenare: 0.5813851348111327, Acuratețe antrenare: 0.7626760563380282
Epoca 1 - Pierdere validare: 0.43248903239145875, Acuratețe validare: 0.8346354166666666
Epoca 2 - Pierdere antrenare: 0.3109395015250209, Acuratețe antrenare: 0.8952464788732394
Epoca 2 - Pierdere validare: 0.42984483554027975, Acuratețe validare: 0.8359375
Epoca 3 - Pierdere antrenare: 0.19572635800380941, Acuratețe antrenare: 0.9328345070422536
Epoca 3 - Pierdere validare: 0.42302563949488103, Acuratețe validare: 0.8580729166666666


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/iulik-pisik/bert_model_vreme/commit/95d9229f89e8feeda254b60d4bebcf3cbfcb9deb', commit_message='Upload tokenizer', commit_description='', oid='95d9229f89e8feeda254b60d4bebcf3cbfcb9deb', pr_url=None, pr_revision=None, pr_num=None)

# Finetune combinat

In [ ]:
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import torch

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Încărcare date
horoscop_train = pd.read_csv('/content/drive/MyDrive/licenta/bert_2classes/filtered_horoscop_train.tsv', sep='\t')
vreme_train = pd.read_csv('/content/drive/MyDrive/licenta/bert_2classes/filtered_vreme_train.tsv', sep='\t')

# Adăugăm o coloană suplimentară pentru a identifica sursa datelor
horoscop_train['category'] = 'horoscop'
vreme_train['category'] = 'vreme'

# Combinăm datele
combined_train = pd.concat([horoscop_train, vreme_train])

# Împărțim datele în seturi de antrenament și validare
train_data, val_data = train_test_split(combined_train, test_size=0.1, random_state=42, stratify=combined_train['class'])


In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup
import torch
from torch.utils.data import DataLoader, TensorDataset

# Inițializare tokenizer
tokenizer = BertTokenizer.from_pretrained('dumitrescustefan/bert-base-romanian-cased-v1')

def tokenize_data(data):
    return tokenizer(data['sentence'].tolist(), padding=True, truncation=True, return_tensors="pt")

# Tokenizare date
train_encodings = tokenize_data(train_data)
val_encodings = tokenize_data(val_data)

# Conversie etichete în tensor
label_map = {'pozitiv': 0, 'negativ': 1}
train_labels = torch.tensor([label_map[label] for label in train_data['class'].tolist()])
val_labels = torch.tensor([label_map[label] for label in val_data['class'].tolist()])

# Creare DataLoader
def create_dataloader(encodings, labels, batch_size=16):
    dataset = TensorDataset(encodings['input_ids'], encodings['attention_mask'], labels)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

train_dataloader = create_dataloader(train_encodings, train_labels)
val_dataloader = create_dataloader(val_encodings, val_labels)

# Inițializare model BERT
model = BertForSequenceClassification.from_pretrained('dumitrescustefan/bert-base-romanian-cased-v1', num_labels=2)
optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 3
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Funcție de antrenare
def train(model, train_dataloader, val_dataloader):
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        total_train_accuracy = 0

        for batch in train_dataloader:
            b_input_ids, b_input_mask, b_labels = batch
            b_input_ids, b_input_mask, b_labels = b_input_ids.to(device), b_input_mask.to(device), b_labels.to(device)

            model.zero_grad()
            outputs = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)
            loss = outputs.loss
            logits = outputs.logits
            total_train_loss += loss.item()

            preds = torch.argmax(logits, dim=1).flatten()
            total_train_accuracy += accuracy_score(b_labels.cpu().numpy(), preds.cpu().numpy())

            loss.backward()
            optimizer.step()
            scheduler.step()

        avg_train_loss = total_train_loss / len(train_dataloader)
        avg_train_accuracy = total_train_accuracy / len(train_dataloader)

        model.eval()
        total_val_loss = 0
        total_val_accuracy = 0

        for batch in val_dataloader:
            b_input_ids, b_input_mask, b_labels = batch
            b_input_ids, b_input_mask, b_labels = b_input_ids.to(device), b_input_mask.to(device), b_labels.to(device)

            with torch.no_grad():
                outputs = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)
                loss = outputs.loss
                logits = outputs.logits
                total_val_loss += loss.item()

                preds = torch.argmax(logits, dim=1).flatten()
                total_val_accuracy += accuracy_score(b_labels.cpu().numpy(), preds.cpu().numpy())

        avg_val_loss = total_val_loss / len(val_dataloader)
        avg_val_accuracy = total_val_accuracy / len(val_dataloader)

        print(f"Epoca {epoch + 1} - Pierdere antrenare: {avg_train_loss}, Acuratețe antrenare: {avg_train_accuracy}")
        print(f"Epoca {epoch + 1} - Pierdere validare: {avg_val_loss}, Acuratețe validare: {avg_val_accuracy}")

# Antrenare model
train(model, train_dataloader, val_dataloader)

# Salvare model
model.save_pretrained("bert_model_v2")

# Upload to Hugging Face Hub
model.push_to_hub("iulik-pisik/bert_model_v2")
tokenizer.push_to_hub("iulik-pisik/bert_model_v2")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dumitrescustefan/bert-base-romanian-cased-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoca 1 - Pierdere antrenare: 0.34518616110015143, Acuratețe antrenare: 0.8449248120300752
Epoca 1 - Pierdere validare: 0.1676716866592566, Acuratețe validare: 0.921875
Epoca 2 - Pierdere antrenare: 0.14212037271343209, Acuratețe antrenare: 0.9455741626794258
Epoca 2 - Pierdere validare: 0.15524954674765468, Acuratețe validare: 0.9401041666666666
Epoca 3 - Pierdere antrenare: 0.06681137200017152, Acuratețe antrenare: 0.9781698564593302
Epoca 3 - Pierdere validare: 0.1578947272791993, Acuratețe validare: 0.9401041666666666


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/iulik-pisik/bert_model_v2/commit/7d46171000601cc9a3c5951c53b78f28a93753ca', commit_message='Upload tokenizer', commit_description='', oid='7d46171000601cc9a3c5951c53b78f28a93753ca', pr_url=None, pr_revision=None, pr_num=None)

# Inferenta

In [1]:
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

In [2]:

# Încărcare model și tokenizer pentru horoscop
# model_horoscop = BertForSequenceClassification.from_pretrained('iulik-pisik/bert_model_horoscop')
# tokenizer_horoscop = BertTokenizer.from_pretrained('iulik-pisik/bert_model_horoscop')

# # Încărcare model și tokenizer pentru vreme
# model_vreme = BertForSequenceClassification.from_pretrained('iulik-pisik/bert_model_vreme')
# tokenizer_vreme = BertTokenizer.from_pretrained('iulik-pisik/bert_model_vreme')

model_combinat = BertForSequenceClassification.from_pretrained('iulik-pisik/bert_model_v2')
tokenizer_combinat = BertTokenizer.from_pretrained('iulik-pisik/bert_model_v2')

config.json:   0%|          | 0.00/719 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/397k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [3]:

# Dispozitiv
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model_horoscop.to(device)
# model_vreme.to(device)
model_combinat.to(device)

# Funcție pentru inferență
def predict(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        predicted_class_id = torch.argmax(logits, dim=-1).item()

    label_map = {0: 'pozitiv', 1: 'negativ'}
    return label_map[predicted_class_id]

# Exemple de text pentru inferență
texts_horoscop = ["Azi vei avea parte de o zi minunată!", "Lucrurile nu vor merge prea bine astăzi."]
texts_vreme = ["Astăzi va fi o zi însorită.", "Se anunță ploi și vreme rece."]


In [4]:
predict(texts_vreme[0], model_combinat, tokenizer_combinat)

'pozitiv'

In [5]:
horoscop_test = pd.read_csv('/content/drive/MyDrive/licenta/bert_2classes/filtered_horoscop_test.tsv', sep='\t')
vreme_test = pd.read_csv('/content/drive/MyDrive/licenta/bert_2classes/filtered_vreme_test.tsv', sep='\t')


In [6]:
test_comb = pd.concat([horoscop_test, vreme_test])

In [19]:
import numpy as np

In [21]:
test_paths = np.array(test_comb['path'])

In [22]:
test_paths

array(['100_13.wav', '100_3.wav', '118_0.wav', ..., '325_24.wav',
       '239_28.wav', '239_10.wav'], dtype=object)

In [24]:
"100_32.wav" in test_paths

False

In [25]:
# Funcție pentru calcularea acurateței și afișarea matricii de confuzie
def evaluate_model(test_data, model, tokenizer):
    texts = test_data['sentence'].tolist()
    true_labels = test_data['class'].tolist()
    predictions = [predict(text, model, tokenizer) for text in texts]

    accuracy = accuracy_score(true_labels, predictions)
    conf_matrix = confusion_matrix(true_labels, predictions, labels=['pozitiv', 'negativ'])
    disp = ConfusionMatrixDisplay(conf_matrix)
    class_report = classification_report(true_labels, predictions, target_names=['pozitiv', 'negativ'])

    print(f"Acuratețea: {accuracy}")
    print("Matricea de confuzie:")
    print(conf_matrix)
    # disp.plot()
    print("Raport de clasificare:")
    print(class_report)

# Evaluare model pentru horoscop
print("Evaluare model horoscop:")
evaluate_model(horoscop_test, model_combinat, tokenizer_combinat)

# Evaluare model pentru vreme
print("Evaluare model vreme:")
evaluate_model(vreme_test, model_combinat, tokenizer_combinat)

print("Evaluare total:")

evaluate_model(test_comb, model_combinat, tokenizer_combinat)


Evaluare model horoscop:
Acuratețea: 0.896414342629482
Matricea de confuzie:
[[159   9]
 [ 17  66]]
Raport de clasificare:
              precision    recall  f1-score   support

     pozitiv       0.88      0.80      0.84        83
     negativ       0.90      0.95      0.92       168

    accuracy                           0.90       251
   macro avg       0.89      0.87      0.88       251
weighted avg       0.90      0.90      0.89       251

Evaluare model vreme:
Acuratețea: 0.9189842805320435
Matricea de confuzie:
[[269  24]
 [ 43 491]]
Raport de clasificare:
              precision    recall  f1-score   support

     pozitiv       0.95      0.92      0.94       534
     negativ       0.86      0.92      0.89       293

    accuracy                           0.92       827
   macro avg       0.91      0.92      0.91       827
weighted avg       0.92      0.92      0.92       827

Evaluare total:
Acuratețea: 0.9137291280148423
Matricea de confuzie:
[[428  33]
 [ 60 557]]
Raport de 

In [27]:
import os

In [42]:
# Creare dicționar pentru ground truth
horoscop_ground_truth = dict(zip(horoscop_test['path'], horoscop_test['class']))
vreme_ground_truth = dict(zip(vreme_test['path'], vreme_test['class']))

# Încărcare și evaluare noi fișiere CSV
# folder_path = '/content/drive/MyDrive/licenta/outputs/whisper/whisper_large'
folder_path = '/content/drive/MyDrive/licenta/outputs/all_data_model/all_data_model_small'
file_names = ["cosmin_stan", "florin_busuioc", "loredana_stefu", "iulia_parlea", "neti_sandu", "urania"]

results = {}
file_sizes = {}

for file_name in file_names:
    file_path = os.path.join(folder_path, file_name + '.csv')
    data = pd.read_csv(file_path)

    model = model_combinat
    tokenizer = tokenizer_combinat

    if file_name in ["neti_sandu", "urania"]:
        ground_truth = horoscop_ground_truth
    else:
        ground_truth = vreme_ground_truth

    data = data[data['prediction'].notnull() & data['prediction'].str.strip().astype(bool)]
    data = data[data['filename'].isin(test_paths)]  # Filtrează doar textele cu filename în test_paths

    texts = data['prediction'].tolist()
    paths = data['filename'].tolist()
    true_labels = [ground_truth[path] for path in paths]  # Nu mai este nevoie de verificarea if path in test_paths
    predictions = [predict(text, model, tokenizer) for text in texts]

    accuracy = accuracy_score(true_labels, predictions)
    results[file_name] = accuracy
    file_sizes[file_name] = len(texts)
    # print(f"Acuratețea pentru {file_name}: {accuracy}")


# Calcularea acurateței generale pentru vreme și horoscop
def calculate_overall_accuracy(results, file_sizes, categories):
    total_correct = 0
    total_count = 0
    for file_name in categories:
        if file_name in results:
            total_correct += results[file_name] * file_sizes[file_name]
            total_count += file_sizes[file_name]
    overall_accuracy = total_correct / total_count if total_count > 0 else 0
    return overall_accuracy

# Definirea categoriilor pentru vreme și horoscop
vreme_files = ["cosmin_stan", "florin_busuioc", "loredana_stefu", "iulia_parlea"]
horoscop_files = ["neti_sandu", "urania"]

# Calcularea acurateței generale
overall_accuracy_vreme = calculate_overall_accuracy(results, file_sizes, vreme_files)
overall_accuracy_horoscop = calculate_overall_accuracy(results, file_sizes, horoscop_files)

print(f"Acuratețea generală pentru vreme: {overall_accuracy_vreme}")
print(f"Acuratețea generală pentru horoscop: {overall_accuracy_horoscop}")
print()

combined_files = vreme_files + horoscop_files
overall_accuracy_combined = calculate_overall_accuracy(results, file_sizes, combined_files)

print(f"Acuratețea generală combinată pentru vreme și horoscop: {overall_accuracy_combined}")


Acuratețea generală pentru vreme: 0.9129383313180169
Acuratețea generală pentru horoscop: 0.8844621513944223

Acuratețea generală combinată pentru vreme și horoscop: 0.9063079777365491


In [ ]:

# Calcularea acurateței grupate după coloana 'name'
def calculate_grouped_accuracy(test_data, model, tokenizer):
    grouped_results = test_data.groupby('name').apply(lambda x: accuracy_score(x['class'].tolist(), [predict(text, model, tokenizer) for text in x['sentence'].tolist()]))
    return grouped_results

# Evaluare pentru horoscop
print("Acuratețea grupată pentru horoscop:")
grouped_accuracy_horoscop = calculate_grouped_accuracy(horoscop_test, model_combinat, tokenizer_combinat)
print(grouped_accuracy_horoscop)

# Evaluare pentru vreme
print("Acuratețea grupată pentru vreme:")
grouped_accuracy_vreme = calculate_grouped_accuracy(vreme_test, model_combinat, tokenizer_combinat)
print(grouped_accuracy_vreme)

# Acuratețea totală pentru seturile noi
total_accuracy_horoscop = accuracy_score(
    [horoscop_ground_truth[path] for path in horoscop_test['path']],
    [predict(text, model_combinat, tokenizer_combinat) for text in horoscop_test['sentence']]
)

total_accuracy_vreme = accuracy_score(
    [vreme_ground_truth[path] for path in vreme_test['path']],
    [predict(text, model_combinat, tokenizer_combinat) for text in vreme_test['sentence']]
)

print(f"Acuratețea totală pentru setul de horoscop: {total_accuracy_horoscop}")
print(f"Acuratețea totală pentru setul de vreme: {total_accuracy_vreme}")

Acuratețea grupată pentru horoscop:
name
Neti Sandu    0.896739
Urania        0.711957
dtype: float64
Acuratețea grupată pentru vreme:
name
Cosmin Stan       0.850515
Florin Busuioc    0.904762
Iulia Parlea      0.877395
Loredana Stefu    0.777344
dtype: float64
Acuratețea totală pentru setul de horoscop: 0.8043478260869565
Acuratețea totală pentru setul de vreme: 0.851380042462845


In [ ]:
def calculate_combined_accuracy(horoscop_test, vreme_test, model, tokenizer):
    combined_texts = horoscop_test['sentence'].tolist() + vreme_test['sentence'].tolist()
    combined_true_labels = horoscop_test['class'].tolist() + vreme_test['class'].tolist()

    combined_predictions = [predict(text, model, tokenizer) for text in combined_texts]

    combined_accuracy = accuracy_score(combined_true_labels, combined_predictions)
    return combined_accuracy


combined_accuracy = calculate_combined_accuracy(horoscop_test, vreme_test, model, tokenizer)
print(f"Acuratețea generală combinată pentru vreme și horoscop: {combined_accuracy}")

Acuratețea generală combinată pentru vreme și horoscop: 0.8381679389312977
